<font color="#CA0032"><h1>Practica B3-T5 - Redes neuronales con entradas heterogeneas</h1></font>

## Prediccion de ventas (Rossmann Store Sales)

**Grupo:** Alonso Diaz - Raul Rodriguez - Piettro Enrico

---

### Indice
1. Objetivo y planteamiento
2. Carga y union de datos
3. Analisis exploratorio (EDA)
4. Las *10 mejores tiendas* (objetivo de evaluacion)
5. Preprocesado e ingenieria de variables
6. Enventanado (ventanas deslizantes por tienda)
7. Particion temporal train / validacion / test interno
8. Modelos de referencia (baselines)
9. Modelo A - solo endogena
10. Modelo B - endogena + exogenas
11. Modelo C - entradas heterogeneas con embeddings
12. Comparativa de metricas (R2)
13. Visualizacion de embeddings
14. Prediccion para 2015-07-31 y fichero de submission
15. Reflexion final

## 1. Objetivo y planteamiento

El objetivo es construir y optimizar un **modelo neuronal con entradas heterogeneas** para predecir las
ventas diarias (`Sales`) de la cadena Rossmann, y evaluarlo mediante el **coeficiente de determinacion R2**
en las **10 mejores tiendas** (las de mayor volumen de ventas).

**Entradas heterogeneas** = combinamos en una sola red senales de naturaleza distinta:
- *endogena*: el pasado reciente de las propias ventas (ventana temporal),
- *exogenas numericas*: distancia a la competencia, etc.,
- *categoricas via embeddings*: tienda, tipo de tienda, surtido, dia de la semana, mes, festivo, promocion.

**Nota metodologica importante.** El fichero `test.csv` (dia 2015-07-31) **no contiene la columna `Sales`**:
el R2 real lo calcula el profesor. Por eso medimos nuestro R2 sobre un **hold-out temporal interno** (las
ultimas semanas del periodo de entrenamiento, que si tienen ventas reales) y, ademas, generamos la prediccion
para 2015-07-31 en el formato de `submission.csv`.

Seguimos el esquema incremental de los notebooks base: **solo endogena -> + exogenas -> + embeddings**.

In [ ]:
# Entorno: poner COLAB=True para ejecutar en Google Colab (descarga los datos del repo de GitHub).
# En local, deja COLAB=False y asegurate de tener la carpeta data/ junto al notebook.
COLAB = False

RUTA_DATA = 'data'  # carpeta con train.zip, store.csv, test.csv, submission.csv

In [ ]:
import numpy as np
import pandas as pd
import zipfile, os
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (Input, Dense, LSTM, GRU, Embedding,
                                     Flatten, concatenate, Dropout)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

SEED = 7
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('TensorFlow', tf.__version__)

In [ ]:
# Si estamos en Colab, clonamos el repo para tener los datos a mano.
if COLAB:
    if not os.path.exists('B3_T5-NN-con-entrdas-hetereogeneas-'):
        !git clone https://github.com/alonsodt/B3_T5-NN-con-entrdas-hetereogeneas-.git
    RUTA_DATA = 'B3_T5-NN-con-entrdas-hetereogeneas-/data'

## 2. Carga y union de datos

Unimos el historico de ventas (`train.csv`) con los metadatos de cada tienda (`store.csv`) por `Store`.

In [ ]:
with zipfile.ZipFile(os.path.join(RUTA_DATA, 'train.zip')) as z:
    train = pd.read_csv(z.open('train.csv'), parse_dates=['Date'], low_memory=False)
store = pd.read_csv(os.path.join(RUTA_DATA, 'store.csv'))
test  = pd.read_csv(os.path.join(RUTA_DATA, 'test.csv'), parse_dates=['Date'])

print('train:', train.shape, '| store:', store.shape, '| test:', test.shape)
print('Rango de fechas en train:', train.Date.min().date(), '->', train.Date.max().date())
print('Fecha(s) en test:', sorted(test.Date.dt.date.unique()))
train.head()

In [ ]:
df = train.merge(store, on='Store', how='left').sort_values(['Store', 'Date']).reset_index(drop=True)
# StateHoliday viene con tipos mezclados ('0' y 0); lo normalizamos a categoria de texto.
df['StateHoliday'] = df['StateHoliday'].astype(str).replace('0', 'n')  # n = no festivo
df.head()

## 3. Analisis exploratorio (EDA)

Buscamos los patrones que la red debera capturar: estacionalidad semanal/anual, efecto de promociones,
dias cerrados, diferencias entre tipos de tienda.

In [ ]:
# Serie agregada de ventas (suma diaria de todas las tiendas)
serie = df.groupby('Date')['Sales'].sum()
plt.figure(figsize=(15, 4))
plt.plot(serie.index, serie.values)
plt.title('Ventas totales diarias (todas las tiendas)')
plt.ylabel('Sales'); plt.grid(); plt.show()

In [ ]:
# Estacionalidad semanal: media de ventas por dia de la semana (1=lunes ... 7=domingo)
# (solo dias abiertos, para no mezclar los ceros de los domingos cerrados)
abierto = df[df.Open == 1]
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
abierto.groupby('DayOfWeek')['Sales'].mean().plot(kind='bar', ax=ax[0], title='Venta media por dia de la semana')
abierto.groupby('Promo')['Sales'].mean().plot(kind='bar', ax=ax[1], title='Venta media con/sin promocion')
plt.show()

In [ ]:
# Dias cerrados y ceros estructurales
print('Filas con Open==0:', (df.Open == 0).sum())
print('De ellas, con Sales>0:', ((df.Open == 0) & (df.Sales > 0)).sum(), '(0 => Open=0 implica Sales=0)')
print()
# Diferencias por tipo de tienda y surtido
print('Venta media por StoreType:')
print(abierto.groupby('StoreType')['Sales'].mean())
print('\nVenta media por Assortment:')
print(abierto.groupby('Assortment')['Sales'].mean())

In [ ]:
# Distribucion de Sales y log1p(Sales): el log normaliza la asimetria (clave para entrenar bien)
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
abierto['Sales'].hist(bins=50, ax=ax[0]); ax[0].set_title('Sales (dias abiertos)')
np.log1p(abierto['Sales']).hist(bins=50, ax=ax[1]); ax[1].set_title('log1p(Sales)')
plt.show()

## 4. Las *10 mejores tiendas* (objetivo de evaluacion)

Interpretamos *las 10 mejores tiendas* como las **10 con mayor volumen total de ventas** en el periodo
disponible. Dejamos el criterio parametrizado por si hubiera que cambiarlo (p. ej. por venta media diaria
o por una lista concreta que indique el profesor).

In [ ]:
ranking = df.groupby('Store')['Sales'].sum().sort_values(ascending=False)
TOP10 = ranking.head(10).index.tolist()
print('Top 10 tiendas por ventas totales:', TOP10)
ranking.head(10)

## 5. Preprocesado e ingenieria de variables

Decisiones (justificadas):
- **Objetivo en escala log**: `y = log1p(Sales)`. Reduce la asimetria y estabiliza el entrenamiento; el R2
  final lo calculamos de vuelta en la escala original (euros).
- **Dias cerrados**: si `Open==0` la venta es 0 por definicion. Entrenamos el modelo solo con dias abiertos,
  y en la prediccion forzamos `Sales=0` cuando la tienda esta cerrada.
- **Variables de calendario** derivadas de la fecha (mes, dia de la semana...). Son *conocidas de antemano*
  para el dia a predecir.
- **Categoricas -> indices enteros** para alimentar capas `Embedding`.
- **Numericas -> estandarizadas** (ajustando el escalador solo con train para no filtrar informacion).

In [ ]:
# Variables de calendario
df['Year']  = df.Date.dt.year
df['Month'] = df.Date.dt.month
df['Day']   = df.Date.dt.day
df['WeekOfYear'] = df.Date.dt.isocalendar().week.astype(int)
# DayOfWeek ya viene 1..7

# CompetitionDistance: imputamos nulos con la mediana y aplicamos log (esta muy sesgada)
df['CompetitionDistance'] = df['CompetitionDistance'].fillna(df['CompetitionDistance'].median())
df['CompDistLog'] = np.log1p(df['CompetitionDistance'])

# Objetivo en escala log
df['y'] = np.log1p(df['Sales'])

In [ ]:
# Codificacion de categoricas a indices enteros (empezando en 0) para los embeddings
cat_cols = ['Store', 'StoreType', 'Assortment', 'StateHoliday', 'DayOfWeek', 'Month']
cat_maps = {}
cat_card = {}  # cardinalidad (nº de categorias) de cada variable
for c in cat_cols:
    cats = sorted(df[c].unique())
    cat_maps[c] = {v: i for i, v in enumerate(cats)}
    df[c + '_idx'] = df[c].map(cat_maps[c]).astype(int)
    cat_card[c] = len(cats)
cat_card

## 6. Enventanado (ventanas deslizantes por tienda)

Implementamos nuestro propio enventanado (autocontenido, sin dependencias externas) porque aqui hay **muchas
tiendas** y las ventanas **no deben cruzar la frontera entre tiendas**.

Para cada dia objetivo `t` de una tienda construimos:
- `X_endo`: los `W` dias previos de `y` (ventas log) -> forma `(W, 1)` (lo desconocido, su pasado),
- las **categoricas y exogenas del dia `t`** (conocidas de antemano: calendario, promo, festivo, tienda...),
- objetivo `y[t]`.

Asi reproducimos el esquema *many-to-one* de los notebooks base, pero con entradas heterogeneas.

In [ ]:
W = 21  # lookback (tamano de ventana): 3 semanas de historia

EXO_NUM = ['Promo', 'SchoolHoliday', 'Promo2', 'CompDistLog']  # exogenas numericas del dia t
CAT_IDX = ['Store_idx', 'StoreType_idx', 'Assortment_idx', 'StateHoliday_idx',
           'DayOfWeek_idx', 'Month_idx']                       # categoricas del dia t

def construir_ventanas(data, W):
    """Genera ventanas por tienda. Devuelve un dict de arrays alineados por fila."""
    endo, y_out, fecha, store_id, abierto_t = [], [], [], [], []
    exo = {c: [] for c in EXO_NUM}
    cat = {c: [] for c in CAT_IDX}
    for _, g in data.groupby('Store'):
        g = g.sort_values('Date')
        yv = g['y'].values
        for i in range(W, len(g)):
            endo.append(yv[i - W:i])           # pasado (W dias)
            y_out.append(yv[i])                # objetivo (dia i)
            fecha.append(g['Date'].values[i])
            store_id.append(g['Store'].values[i])
            abierto_t.append(g['Open'].values[i])
            for c in EXO_NUM:
                exo[c].append(g[c].values[i])
            for c in CAT_IDX:
                cat[c].append(g[c].values[i])
    out = {
        'endogena': np.array(endo)[..., None].astype('float32'),  # (N, W, 1)
        'y': np.array(y_out).astype('float32'),
        'fecha': np.array(fecha),
        'store': np.array(store_id),
        'open': np.array(abierto_t),
        'num': np.stack([np.array(exo[c], dtype='float32') for c in EXO_NUM], axis=1),  # (N, n_num)
    }
    for c in CAT_IDX:
        out[c] = np.array(cat[c]).astype('int32')
    return out

# Entrenamos solo con dias ABIERTOS como objetivo (los cerrados son 0 por definicion),
# pero la ventana de pasado conserva todos los dias (incluidos ceros de domingos).
dataset = construir_ventanas(df, W)
# filtramos objetivos de dias cerrados:
mask_open = dataset['open'] == 1
for k in list(dataset.keys()):
    dataset[k] = dataset[k][mask_open]
print('Total de muestras (dias abiertos):', dataset['y'].shape[0])
print('Forma de la endogena:', dataset['endogena'].shape, '| forma de num:', dataset['num'].shape)

## 7. Particion temporal train / validacion / test interno

Particion **temporal** (nunca aleatoria en series temporales):
- **Test interno**: ultimas 6 semanas con datos reales (`2015-06-06` a `2015-07-17`). Es nuestro proxy del
  test del profesor, y sobre el calculamos el R2 (en particular el de las 10 mejores tiendas).
- **Validacion**: las 6 semanas anteriores (para *early stopping* / seleccion de modelo).
- **Train**: todo lo anterior.

In [ ]:
FECHA_TEST_INI = np.datetime64('2015-06-06')
FECHA_VAL_INI  = np.datetime64('2015-04-25')

fechas = dataset['fecha']
idx_train = np.where(fechas < FECHA_VAL_INI)[0]
idx_val   = np.where((fechas >= FECHA_VAL_INI) & (fechas < FECHA_TEST_INI))[0]
idx_test  = np.where(fechas >= FECHA_TEST_INI)[0]
print('train:', len(idx_train), '| val:', len(idx_val), '| test:', len(idx_test))

In [ ]:
# Estandarizamos la endogena y las numericas con estadisticos calculados SOLO en train.
esc_endo = StandardScaler().fit(dataset['endogena'][idx_train].reshape(-1, 1))
esc_num  = StandardScaler().fit(dataset['num'][idx_train])

def escala_endo(a):
    s = a.shape
    return esc_endo.transform(a.reshape(-1, 1)).reshape(s).astype('float32')

endo_s = escala_endo(dataset['endogena'])
num_s  = esc_num.transform(dataset['num']).astype('float32')

In [ ]:
def arma_inputs(idx, usar_exo=True, usar_cat=True):
    """Construye el diccionario de entradas para el subconjunto idx segun el tipo de modelo."""
    X = {'endogena': endo_s[idx]}
    if usar_exo:
        X['num'] = num_s[idx]
    if usar_cat:
        for c in CAT_IDX:
            X[c] = dataset[c][idx]
    return X

y_train = dataset['y'][idx_train]
y_val   = dataset['y'][idx_val]
y_test  = dataset['y'][idx_test]

# Ventas reales (escala original) para calcular R2 de forma interpretable
ventas_test = np.expm1(y_test)
store_test  = dataset['store'][idx_test]

## 8. Modelos de referencia (baselines)

Antes de las redes, fijamos referencias para contextualizar el R2:
- **Persistente a 7 dias**: la venta de un dia = la del mismo dia de la semana anterior.
- **Media historica por (tienda, dia de la semana)** calculada en train.

Una red solo aporta valor si supera claramente a estos baselines.

In [ ]:
# Baseline media por (tienda, dia de la semana) usando train (escala original)
df_train_open = df[(df.Date < FECHA_TEST_INI) & (df.Open == 1)]
media_td = df_train_open.groupby(['Store', 'DayOfWeek'])['Sales'].mean()

# Recuperamos el DayOfWeek real del test interno a partir de los indices guardados:
inv_dow = {v: k for k, v in cat_maps['DayOfWeek'].items()}
dow_test = np.array([inv_dow[i] for i in dataset['DayOfWeek_idx'][idx_test]])

pred_media = np.array([media_td.get((s, d), df_train_open['Sales'].mean())
                       for s, d in zip(store_test, dow_test)])

def r2_top10(ventas_reales, pred, stores):
    """R2 global y R2 restringido a las 10 mejores tiendas."""
    m = np.isin(stores, TOP10)
    return r2_score(ventas_reales, pred), r2_score(ventas_reales[m], pred[m])

g, t = r2_top10(ventas_test, pred_media, store_test)
print(f'Baseline media (tienda,dia): R2 global={g:.4f} | R2 top10={t:.4f}')

## 9-11. Construccion y entrenamiento de los modelos

Definimos una funcion `construir_modelo(...)` que, segun los flags, genera las tres arquitecturas del
esquema incremental:

| Modelo | Endogena | Exogenas num. | Embeddings categoricos |
|--------|:--------:|:-------------:|:----------------------:|
| **A** solo endogena | si | no | no |
| **B** + exogenas | si | si | no |
| **C** entradas heterogeneas | si | si | **si** |

En todos, una capa recurrente (LSTM/GRU) resume el pasado de la serie; el resto de entradas se concatenan
con ese resumen antes de las capas densas finales.

In [ ]:
def construir_modelo(usar_exo=False, usar_cat=False, rnn='LSTM', units=32, dim_emb=8):
    inputs = []
    inp_endo = Input(shape=(W, 1), name='endogena'); inputs.append(inp_endo)
    capa_rnn = LSTM if rnn == 'LSTM' else GRU
    h = capa_rnn(units)(inp_endo)
    ramas = [h]

    if usar_cat:
        for c in CAT_IDX:
            card = cat_card[c.replace('_idx', '')]
            d = min(dim_emb, max(2, card // 2))  # dim. embedding adaptada a la cardinalidad
            inp = Input(shape=(1,), name=c); inputs.append(inp)
            e = Flatten()(Embedding(input_dim=card, output_dim=d)(inp))
            ramas.append(e)

    if usar_exo:
        inp_num = Input(shape=(len(EXO_NUM),), name='num'); inputs.append(inp_num)
        ramas.append(inp_num)

    x = concatenate(ramas) if len(ramas) > 1 else ramas[0]
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.2)(x)
    x = Dense(32, activation='relu')(x)
    out = Dense(1)(x)
    model = Model(inputs=inputs, outputs=out)
    model.compile(loss='mse', optimizer='adam', metrics=['mae'])
    return model

In [ ]:
def entrenar(model, usar_exo, usar_cat, epochs=40, batch_size=512, nombre='modelo'):
    Xtr = arma_inputs(idx_train, usar_exo, usar_cat)
    Xva = arma_inputs(idx_val,   usar_exo, usar_cat)
    ruta = f'{nombre}.keras'
    cbs = [ModelCheckpoint(ruta, monitor='val_loss', save_best_only=True),
           EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)]
    hist = model.fit(Xtr, y_train, validation_data=(Xva, y_val),
                     epochs=epochs, batch_size=batch_size, verbose=2, callbacks=cbs)
    return hist

def evaluar(model, usar_exo, usar_cat):
    Xte = arma_inputs(idx_test, usar_exo, usar_cat)
    pred_log = model.predict(Xte, verbose=0).flatten()
    pred = np.expm1(pred_log)               # de vuelta a euros
    pred = np.clip(pred, 0, None)
    return r2_top10(ventas_test, pred, store_test)

def grafica_hist(hist, titulo):
    plt.figure(figsize=(9, 3))
    plt.plot(hist.history['loss'], label='train')
    plt.plot(hist.history['val_loss'], label='val')
    plt.title(titulo); plt.xlabel('epoca'); plt.ylabel('mse (log)'); plt.legend(); plt.grid(); plt.show()

### 9. Modelo A - solo endogena

In [ ]:
tf.random.set_seed(SEED)
modelA = construir_modelo(usar_exo=False, usar_cat=False)
histA = entrenar(modelA, usar_exo=False, usar_cat=False, nombre='modeloA')
grafica_hist(histA, 'Modelo A (solo endogena)')
r2A_g, r2A_t = evaluar(modelA, False, False)
print(f'Modelo A -> R2 global={r2A_g:.4f} | R2 top10={r2A_t:.4f}')

### 10. Modelo B - endogena + exogenas

In [ ]:
tf.random.set_seed(SEED)
modelB = construir_modelo(usar_exo=True, usar_cat=False)
histB = entrenar(modelB, usar_exo=True, usar_cat=False, nombre='modeloB')
grafica_hist(histB, 'Modelo B (endogena + exogenas)')
r2B_g, r2B_t = evaluar(modelB, True, False)
print(f'Modelo B -> R2 global={r2B_g:.4f} | R2 top10={r2B_t:.4f}')

### 11. Modelo C - entradas heterogeneas con embeddings

Es el modelo central de la practica: anade un `Embedding` por cada variable categorica (tienda, tipo,
surtido, festivo, dia de la semana, mes) y los concatena con el resumen recurrente de la endogena y las
exogenas numericas.

In [ ]:
tf.random.set_seed(SEED)
modelC = construir_modelo(usar_exo=True, usar_cat=True)
modelC.summary()
histC = entrenar(modelC, usar_exo=True, usar_cat=True, nombre='modeloC')
grafica_hist(histC, 'Modelo C (entradas heterogeneas + embeddings)')
r2C_g, r2C_t = evaluar(modelC, True, True)
print(f'Modelo C -> R2 global={r2C_g:.4f} | R2 top10={r2C_t:.4f}')

## 12. Comparativa de metricas (R2)

Resumen de R2 en el test interno (escala original de ventas), global y restringido a las 10 mejores tiendas.

In [ ]:
resumen = pd.DataFrame({
    'Modelo': ['Baseline media (tienda,dia)', 'A - solo endogena',
               'B - + exogenas', 'C - entradas heterogeneas (embeddings)'],
    'R2 global': [r2_top10(ventas_test, pred_media, store_test)[0], r2A_g, r2B_g, r2C_g],
    'R2 top10':  [r2_top10(ventas_test, pred_media, store_test)[1], r2A_t, r2B_t, r2C_t],
})
resumen

## 13. Visualizacion de embeddings

Los embeddings aprendidos colocan categorias similares cerca en el espacio latente. Visualizamos el del
dia de la semana (esperamos ver agrupados los laborables frente al fin de semana).

In [ ]:
emb_layer = [l for l in modelC.layers if l.name.startswith('embedding')]
# El primer embedding corresponde a la primera categorica de CAT_IDX (Store). Buscamos el de DayOfWeek.
# Reconstruimos por orden de creacion = orden de CAT_IDX.
orden_cat = CAT_IDX
emb_por_cat = dict(zip(orden_cat, emb_layer))

W_dow = emb_por_cat['DayOfWeek_idx'].get_weights()[0]
print('Embedding DayOfWeek shape:', W_dow.shape)
if W_dow.shape[1] >= 2:
    plt.figure(figsize=(5, 4))
    nombres = ['lun', 'mar', 'mie', 'jue', 'vie', 'sab', 'dom']
    plt.scatter(W_dow[:, 0], W_dow[:, 1])
    for i, n in enumerate(nombres[:W_dow.shape[0]]):
        plt.text(W_dow[i, 0], W_dow[i, 1], n)
    plt.title('Embedding del dia de la semana'); plt.grid(); plt.show()

## 14. Prediccion para 2015-07-31 y fichero de submission

Para el dia objetivo del profesor (`2015-07-31`) no disponemos de las ventas de los dias inmediatamente
anteriores (train acaba el 2015-07-17). Hacemos por tanto una **prediccion multi-step autoregresiva**: a
partir del 2015-07-17 avanzamos dia a dia hasta el 2015-07-31, realimentando la ventana con las propias
predicciones y usando el calendario/promo real de cada dia futuro (que si conocemos).

Las tiendas cerradas ese dia (`Open==0`) se fuerzan a `Sales=0`.

In [ ]:
# Preparamos el test del profesor con las mismas transformaciones
test_p = test.merge(store, on='Store', how='left').copy()
test_p['StateHoliday'] = test_p['StateHoliday'].astype(str).replace('0', 'n')
test_p['Month'] = test_p.Date.dt.month
test_p['CompetitionDistance'] = test_p['CompetitionDistance'].fillna(df['CompetitionDistance'].median())
test_p['CompDistLog'] = np.log1p(test_p['CompetitionDistance'])
for c in ['Store', 'StoreType', 'Assortment', 'StateHoliday', 'DayOfWeek', 'Month']:
    test_p[c + '_idx'] = test_p[c].map(cat_maps[c]).fillna(0).astype(int)

def predecir_dia(store_row_idx_df, hist_log):
    """Predice un dia para una fila de test_p dada la historia log (longitud >= W)."""
    fila = test_p.iloc[store_row_idx_df]
    X = {'endogena': escala_endo(np.array(hist_log[-W:])[None, :, None]),
         'num': esc_num.transform([[fila['Promo'], fila['SchoolHoliday'], fila['Promo2'], fila['CompDistLog']]]).astype('float32')}
    for c in CAT_IDX:
        X[c] = np.array([fila[c]], dtype='int32')
    return modelC.predict(X, verbose=0).flatten()[0]

preds_finales = {}
for j in range(len(test_p)):
    s = test_p.iloc[j]['Store']
    if test_p.iloc[j]['Open'] == 0:
        preds_finales[test_p.iloc[j]['Id']] = 0.0
        continue
    hist = df[df.Store == s].sort_values('Date')['y'].values.tolist()  # historia log hasta 2015-07-17
    # avanzamos del 2015-07-18 al 2015-07-31 (14 pasos) de forma autoregresiva
    for _ in range(14):
        p = predecir_dia(j, hist)
        hist.append(p)
    preds_finales[test_p.iloc[j]['Id']] = float(np.clip(np.expm1(hist[-1]), 0, None))

submission = pd.DataFrame({'Id': list(preds_finales.keys()), 'Sales': list(preds_finales.values())})
submission.to_csv('submission_grupo.csv', index=False)
print('Guardado submission_grupo.csv'); submission.head()

> **Aviso**: la prediccion autoregresiva usa el mismo calendario para los 14 pasos intermedios de forma
> aproximada (solo es exacto el dia objetivo). Para una entrega mas fina se puede construir el calendario
> real de cada dia intermedio. El R2 que reportamos para comparar arquitecturas es el del **test interno**
> (seccion 12), que si tiene ventas reales.

## 15. Reflexion final

*(A completar por el grupo con los numeros concretos tras ejecutar.)*

- **Aporte de las entradas heterogeneas.** Comparar el salto de R2 del modelo A (solo endogena) al B
  (+ exogenas) y al C (+ embeddings). La hipotesis es que los embeddings de `Store`, `Promo`, `DayOfWeek` y
  `StateHoliday` aportan la mayor parte de la mejora, porque capturan el nivel propio de cada tienda y los
  picos de promocion/festivos que la endogena sola no anticipa.
- **Modelo global vs. por tienda.** Un unico modelo con embedding de `Store` aprende las 1.115 series a la
  vez, compartiendo patrones (estacionalidad semanal, efecto promo) y especializandose por tienda via el
  embedding. Es mas robusto que entrenar una red por tienda, sobre todo en tiendas con poco historico.
- **Limitaciones.** (1) El test real es a 2 semanas vista sin ventas intermedias -> la prediccion
  multi-step acumula error; (2) no usamos `Customers` (en un escenario real no se conoce de antemano);
  (3) los dias cerrados se tratan de forma deterministica.
- **Mejoras futuras.** Tunear `W`, unidades y dimension de embeddings; probar GRU vs LSTM; anadir variables
  de competencia/Promo2 mas elaboradas; ensemble de varias semillas; o un modelo seq2seq que prediga los 14
  dias de golpe en lugar de autoregresivamente.